In [1]:
import torch
import optuna
from dicee.config import Namespace
from dicee.executer import Execute

import os
import sys
import logging
from contextlib import contextmanager

c:\Users\zalak\miniconda3\envs\dicee_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
args = Namespace()

# Point to your custom absolute folder location
args.dataset_dir = r"C:\my-code\Robust Embedding\dicee\uml_dataset"

# TransE
#### MRR - 0.7256

In [21]:
args.model = "TransE"

args.scoring_technique = "AllvsAll"
args.num_negatives_per_positive = 10

# HyperParams Parameters
args.num_epochs = 150
args.embedding_dim = 128
args.lr = 0.01

args.trainer = "torchCPUTrainer"
args.eval_model = 'test'
args.path_to_store_single_run = "fresh_umls_transe_run"

print("Intialize Pipeline")
executor = Execute(args)
executor.start()


Seed set to 0


Intialize Pipeline
Deleting existing directory: fresh_umls_transe_run
Start time:2026-06-18 13:01:15.898828
Creating knowledge graph...
*** Read or Load Knowledge Graph  ***
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\test.txt with Pandas ***
read_with_pandas took 0.0167 seconds | Current Memory Usage  302.98 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\train.txt with Pandas ***
read_with_pandas took 0.0149 seconds | Current Memory Usage  303.58 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\valid.txt with Pandas ***
read_with_pandas took 0.0036 seconds | Current Memory Usage  303.58 in MB
Adding reciprocal triples to Train, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Validation, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Test, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Concatenating data to build vocabulary...
Creating a mapping from entities to integer indexes...
Indexing tra


  | Name                             | Type              | Params | Mode  | FLOPs
---------------------------------------------------------------------------------------
0 | loss                             | BCEWithLogitsLoss | 0      | train | 0    
1 | normalize_head_entity_embeddings | IdentityClass     | 0      | train | 0    
2 | normalize_relation_embeddings    | IdentityClass     | 0      | train | 0    
3 | normalize_tail_entity_embeddings | IdentityClass     | 0      | train | 0    
4 | hidden_normalizer                | IdentityClass     | 0      | train | 0    
5 | input_dp_ent_real                | Dropout           | 0      | train | 0    
6 | input_dp_rel_real                | Dropout           | 0      | train | 0    
7 | hidden_dropout                   | Dropout           | 0      | train | 0    
8 | entity_embeddings                | Embedding         | 17.3 K | train | 0    
9 | relation_embeddings              | Embedding         | 11.8 K | train | 0    
---------

Number of datapoints: 12420
Took 0.3578 secs | Current Memory Usage 305.88518 MB
Took 0.3582 secs | Current Memory Usage 305.88518 MB
Initializing Dataloader...	Took 0.0005 secs | Current Memory Usage 305.88518 MB
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0.0
)

Training is starting 2026-06-18 13:01:16.573766...
NumOfDataPoints:12420 | NumOfEpochs:150 | LearningRate:0.01 | BatchSize:1024 | EpochBatchsize:13


Epoch:150: 100%|██████████| 150/150 [05:03<00:00,  2.02s/it, loss_step=0.01135, loss_epoch=0.01432]


Training Runtime: 5.068 minutes.

*** Save Trained Model ***
Took 0.0162 secs | Current Memory Usage 469.57363 MB
Total Runtime: 304.357 seconds
Evaluate TransE on Test set: Evaluate TransE on Test set
{'H@1': 0.6051437216338881, 'H@3': 0.8146747352496218, 'H@10': 0.9236006051437217, 'MRR': 0.7268522329119556}
Total Runtime: 304.832 seconds


{'num_train_triples': 10432,
 'num_entities': 135,
 'num_relations': 92,
 'max_length_subword_tokens': None,
 'runtime_kg_loading': 0.23897933959960938,
 'EstimatedSizeMB': 0.0277099609375,
 'NumParam': 29056,
 'path_experiment_folder': 'fresh_umls_transe_run',
 'Runtime': 304.83183240890503,
 'Test': {'H@1': 0.6051437216338881,
  'H@3': 0.8146747352496218,
  'H@10': 0.9236006051437217,
  'MRR': 0.7268522329119556}}

In [29]:
def objective(trial):

    # 1. Define the search boundaries for your hyperparameters
    args = Namespace()
    args.dataset_dir = r"C:\my-code\Robust Embedding\dicee\uml_dataset"
    args.model = "TransE"
    args.scoring_technique = "AllvsAll"
    args.trainer = "torchCPUTrainer"
    
    # Let Optuna pick values for this trial
    args.embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])
    args.lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    args.num_negatives_per_positive = trial.suggest_int("num_negatives_per_positive", 5, 20)
    args.num_epochs = trial.suggest_int("num_epochs", 50, 200)
    
    # Mute intermediate outputs to keep your terminal clean
    args.path_to_store_single_run = f"optuna_trial_{trial.number}"
    args.eval_model = 'val'  # ALWAYS optimize on validation set, NOT test set!

    try:
        # 2. Run the pipeline execution step
        executor = Execute(args)
        
        # 3. Intercept metrics directly from the trainer's final state report
        report = executor.start()
        
        # Access validation MRR from the report dictionary
        val_mrr = report.get('MRR', 0.0)
        return val_mrr
        
    except Exception as e:
        # Handle bad combinations cleanly without crashing the complete study loop
        return 0.0

# 4. Initialize and launch the optimization study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10) # Set to 20 or 30 for a comprehensive search

# 5. Print out the ultimate settings found
print("\n🏆 Optimization Complete! Best Hyperparameters:")
print(study.best_params)
print(f"Achieved Validation MRR: {study.best_value}")

[I 2026-06-18 13:14:58,016] A new study created in memory with name: no-name-b8f6a44a-4c15-4b5e-aad1-bd4f5fd87e91
Seed set to 0


Start time:2026-06-18 13:14:58.079742
Creating knowledge graph...
*** Read or Load Knowledge Graph  ***
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\test.txt with Pandas ***
read_with_pandas took 0.0673 seconds | Current Memory Usage  88.019 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\train.txt with Pandas ***
read_with_pandas took 0.0146 seconds | Current Memory Usage  88.834 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\valid.txt with Pandas ***
read_with_pandas took 0.0031 seconds | Current Memory Usage  88.924 in MB
Adding reciprocal triples to Train, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Validation, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Test, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Concatenating data to build vocabulary...
Creating a mapping from entities to integer indexes...
Indexing train data with shape (10432, 3)...
Indexing valid data with shape (1304,


  | Name                             | Type              | Params | Mode  | FLOPs
---------------------------------------------------------------------------------------
0 | loss                             | BCEWithLogitsLoss | 0      | train | 0    
1 | normalize_head_entity_embeddings | IdentityClass     | 0      | train | 0    
2 | normalize_relation_embeddings    | IdentityClass     | 0      | train | 0    
3 | normalize_tail_entity_embeddings | IdentityClass     | 0      | train | 0    
4 | hidden_normalizer                | IdentityClass     | 0      | train | 0    
5 | input_dp_ent_real                | Dropout           | 0      | train | 0    
6 | input_dp_rel_real                | Dropout           | 0      | train | 0    
7 | hidden_dropout                   | Dropout           | 0      | train | 0    
8 | entity_embeddings                | Embedding         | 8.6 K  | train | 0    
9 | relation_embeddings              | Embedding         | 5.9 K  | train | 0    
---------

Number of datapoints: 12420
Took 0.3598 secs | Current Memory Usage 101.28998 MB
Took 0.3602 secs | Current Memory Usage 101.28998 MB
Initializing Dataloader...	Took 0.0017 secs | Current Memory Usage 101.49888 MB
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.020654299118760985
    maximize: False
    weight_decay: 0.0
)

Training is starting 2026-06-18 13:14:58.801402...
NumOfDataPoints:12420 | NumOfEpochs:50 | LearningRate:0.020654299118760985 | BatchSize:1024 | EpochBatchsize:13


Epoch:50: 100%|██████████| 50/50 [01:03<00:00,  1.27s/it, loss_step=0.03115, loss_epoch=0.01588]


...

Start time:2026-06-18 13:24:14.187451
Creating knowledge graph...
*** Read or Load Knowledge Graph  ***
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\test.txt with Pandas ***
read_with_pandas took 0.0067 seconds | Current Memory Usage  347.08 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\train.txt with Pandas ***
read_with_pandas took 0.0165 seconds | Current Memory Usage  347.34 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\valid.txt with Pandas ***
read_with_pandas took 0.0086 seconds | Current Memory Usage  347.42 in MB
Adding reciprocal triples to Train, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Validation, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Test, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Concatenating data to build vocabulary...
Creating a mapping from entities to integer indexes...
Indexing train data with shape (10432, 3)...
Indexing valid data with shape (1304,


  | Name                             | Type              | Params | Mode  | FLOPs
---------------------------------------------------------------------------------------
0 | loss                             | BCEWithLogitsLoss | 0      | train | 0    
1 | normalize_head_entity_embeddings | IdentityClass     | 0      | train | 0    
2 | normalize_relation_embeddings    | IdentityClass     | 0      | train | 0    
3 | normalize_tail_entity_embeddings | IdentityClass     | 0      | train | 0    
4 | hidden_normalizer                | IdentityClass     | 0      | train | 0    
5 | input_dp_ent_real                | Dropout           | 0      | train | 0    
6 | input_dp_rel_real                | Dropout           | 0      | train | 0    
7 | hidden_dropout                   | Dropout           | 0      | train | 0    
8 | entity_embeddings                | Embedding         | 17.3 K | train | 0    
9 | relation_embeddings              | Embedding         | 11.8 K | train | 0    
---------

Number of datapoints: 12420
Took 0.2899 secs | Current Memory Usage 349.01606 MB
Took 0.2901 secs | Current Memory Usage 349.01606 MB
Initializing Dataloader...	Took 0.0001 secs | Current Memory Usage 349.01606 MB
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.07907672385058845
    maximize: False
    weight_decay: 0.0
)

Training is starting 2026-06-18 13:24:14.660415...
NumOfDataPoints:12420 | NumOfEpochs:56 | LearningRate:0.07907672385058845 | BatchSize:1024 | EpochBatchsize:13


Epoch:32:  55%|█████▌    | 31/56 [01:10<00:57,  2.28s/it, loss_step=0.01580, loss_epoch=0.01687]
[W 2026-06-18 13:25:25,352] Trial 5 failed with parameters: {'embedding_dim': 128, 'lr': 0.07907672385058845, 'num_negatives_per_positive': 8, 'num_epochs': 56} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\zalak\miniconda3\envs\dicee_env\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\zalak\AppData\Local\Temp\ipykernel_31012\3284420109.py", line 25, in objective
    report = executor.start()
             ^^^^^^^^^^^^^^^^
  File "c:\Users\zalak\miniconda3\envs\dicee_env\Lib\site-packages\dicee\executer.py", line 357, in start
    self.trained_model, form_of_labelling = self.trainer.start(knowledge_graph=self.knowledge_graph)
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

KeyboardInterrupt: 

# CKeci
#### CNN Family
#### MRR - 0.8432

In [ ]:
# 1. Setup configurations
args = Namespace()
args.dataset_dir = r"C:\my-code\Robust Embedding\dicee\uml_dataset"
args.trainer = "torchCPUTrainer"

# --- HYPERPARAMETERS ---
args.model = "CKeci"
args.scoring_technique = "1vsAll"
args.embedding_dim = 128
args.lr = 0.01
args.num_negatives_per_positive WERGRWFDDFHR$&&$&%$%$$&OJRSTLZNLRTJZNLÖRTJKNTÖLläkhjklgnlädfkjklöfdheioföljhouhdjkdkjbvdbjk-----------------------

args.path_to_store_single_run = "standalone_distmult_default"
args.eval_model = 'test'  # Evaluates on the test split when finished

print("🚀 Launching DistMult (Showing default output logs)...")

# 2. Execute the model with standard printing
executor = Execute(args)
report = executor.start()

# 3. Print the raw report object at the very end
print("\n Raw Report Summary Object:")
print(report)

Seed set to 0


🚀 Launching DistMult (Showing default output logs)...
Deleting existing directory: standalone_distmult_default
Start time:2026-06-20 00:22:04.695571
Creating knowledge graph...
*** Read or Load Knowledge Graph  ***
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\test.txt with Pandas ***
read_with_pandas took 0.0048 seconds | Current Memory Usage  301.51 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\train.txt with Pandas ***
read_with_pandas took 0.0124 seconds | Current Memory Usage  301.76 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\valid.txt with Pandas ***
read_with_pandas took 0.0022 seconds | Current Memory Usage  301.77 in MB
Adding reciprocal triples to Train, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Validation, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Test, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Concatenating data to build vocabulary...
Creating a mapping from en

Epoch:250: 100%|██████████| 250/250 [02:10<00:00,  1.92it/s, loss_step=0.03055, loss_epoch=0.03396]


Training Runtime: 2.176 minutes.

*** Save Trained Model ***
Took 0.0022 secs | Current Memory Usage 313.30714 MB
Total Runtime: 130.628 seconds
Evaluate CKeci on Test set: Evaluate CKeci on Test set
{'H@1': 0.7692889561270801, 'H@3': 0.897125567322239, 'H@10': 0.9583963691376702, 'MRR': 0.843209280887596}
Total Runtime: 130.933 seconds

 Raw Report Summary Object:
{'num_train_triples': 10432, 'num_entities': 135, 'num_relations': 92, 'max_length_subword_tokens': None, 'runtime_kg_loading': 0.0621037483215332, 'EstimatedSizeMB': 0.027710914611816406, 'NumParam': 29057, 'path_experiment_folder': 'standalone_distmult_default', 'Runtime': 130.93257427215576, 'Test': {'H@1': 0.7692889561270801, 'H@3': 0.897125567322239, 'H@10': 0.9583963691376702, 'MRR': 0.843209280887596}}


# QMult
#### Hypercomplex Family
#### MRR - 0.8998

In [38]:
# 1. Setup configurations
args = Namespace()
args.dataset_dir = r"C:\my-code\Robust Embedding\dicee\uml_dataset"
args.trainer = "torchCPUTrainer"

# --- HYPERPARAMETERS ---
args.model = "QMult"
args.scoring_technique = "1vsAll"
args.embedding_dim = 128
args.lr = 0.01
args.num_negatives_per_positive = 20
args.num_epochs = 250
args.weight_decay = 1e-5
# -----------------------

args.path_to_store_single_run = "standalone_distmult_default"
args.eval_model = 'test'  # Evaluates on the test split when finished

print("🚀 Launching DistMult (Showing default output logs)...")

# 2. Execute the model with standard printing
executor = Execute(args)
report = executor.start()

# 3. Print the raw report object at the very end
print("\n Raw Report Summary Object:")
print(report)

Seed set to 0


🚀 Launching DistMult (Showing default output logs)...
Deleting existing directory: standalone_distmult_default
Start time:2026-06-20 00:33:53.051264
Creating knowledge graph...
*** Read or Load Knowledge Graph  ***
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\test.txt with Pandas ***
read_with_pandas took 0.0045 seconds | Current Memory Usage  305.7 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\train.txt with Pandas ***
read_with_pandas took 0.0127 seconds | Current Memory Usage  305.7 in MB
*** Reading C:\my-code\Robust Embedding\dicee\uml_dataset\valid.txt with Pandas ***
read_with_pandas took 0.0031 seconds | Current Memory Usage  305.72 in MB
Adding reciprocal triples to Train, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Validation, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Adding reciprocal triples to Test, e.g. KG:= (s, p, o) union (o, p_inverse, s)
Concatenating data to build vocabulary...
Creating a mapping from enti

Epoch:250: 100%|██████████| 250/250 [02:11<00:00,  1.91it/s, loss_step=0.02798, loss_epoch=0.03137]


Training Runtime: 2.185 minutes.

*** Save Trained Model ***
Took 0.0019 secs | Current Memory Usage 317.96429 MB
Total Runtime: 131.187 seconds
Evaluate QMult on Test set: Evaluate QMult on Test set
{'H@1': 0.8275340393343419, 'H@3': 0.9667170953101362, 'H@10': 0.9863842662632375, 'MRR': 0.8998013538135746}
Total Runtime: 131.479 seconds

 Raw Report Summary Object:
{'num_train_triples': 10432, 'num_entities': 135, 'num_relations': 92, 'max_length_subword_tokens': None, 'runtime_kg_loading': 0.0730741024017334, 'EstimatedSizeMB': 0.0277099609375, 'NumParam': 29056, 'path_experiment_folder': 'standalone_distmult_default', 'Runtime': 131.47947096824646, 'Test': {'H@1': 0.8275340393343419, 'H@3': 0.9667170953101362, 'H@10': 0.9863842662632375, 'MRR': 0.8998013538135746}}
